# M1: XGBoost + Random Forest + SHAP on CIC (Colab)
Owner: Joshua Tan. A **CPU runtime is enough**. **Runtime → Run all**.

Same data setup as the M2 notebook: the 5 CIC PCAP zips in one Drive folder (default `MyDrive/ICT3214 datasets`).
If the M2 notebook already built the CIC table on your Drive (`MyDrive/leakmate/data/processed/cic/queries.parquet`), it is reused.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/leakmate'
BRANCH = 'main'
!mkdir -p $DRIVE/results

In [ ]:
%cd /content
# Private repo: add a GitHub token in Colab Secrets (key icon) named GH_TOKEN, notebook access ON.
from google.colab import userdata
try:
    tok = userdata.get('GH_TOKEN') + '@'
except Exception:
    tok = ''   # public repo / no secret
!rm -rf leakmate && git clone -q -b $BRANCH https://{tok}github.com/TerminallyStoned/leakmate.git
%cd /content/leakmate
!pip -q install -r requirements-m1.txt

In [ ]:
# Build the CIC query table from the zips on Drive (cached on Drive after the first run)
import os, glob, re, zipfile
RAW_DIR = f'/content/drive/MyDrive/ICT3214 datasets'   # folder holding the 5 CIC zips
CIC = f'{DRIVE}/data/processed/cic/queries.parquet'

def target(fname):
    n = re.sub(r'[^a-z]', '', os.path.basename(fname).lower().replace('.zip', ''))
    for key, sub in [('lightattack', 'light_attacks'), ('heavyattack', 'heavy_attacks'),
                     ('lightbenign', 'light_benign'), ('heavybenign', 'heavy_benign'), ('benign', 'benign_main')]:
        if n.startswith(key):
            return sub
    return None

if not os.path.exists(CIC):
    zips = glob.glob(f'{RAW_DIR}/*.zip')
    found = {z: target(z) for z in zips}
    print(found)
    assert sorted(v for v in found.values() if v) == sorted(
        ['benign_main', 'light_benign', 'light_attacks', 'heavy_benign', 'heavy_attacks']), 'missing or extra zips'
    for z, sub in found.items():
        dest = f'data/raw/cic/{sub}'
        zipfile.ZipFile(z).extractall(dest)
        for inner in glob.glob(f'{dest}/*.zip'):   # in case a zip holds another zip
            zipfile.ZipFile(inner).extractall(dest)
    !find data/raw/cic -name '*.pcap' | sort
    !python scripts/cic_to_queries.py --raw data/raw/cic --out "$CIC"
DATA, EXP = CIC, 'E2'
import pandas as pd
df = pd.read_parquet(DATA); print(df.shape); print(df.groupby(['split', 'label']).size())

## 1. M1: XGBoost (per-query + per-host features, SHAP)

In [ ]:
!python -m models.m1_run --model m1 --exp $EXP --data $DATA --save-model models/weights/m1.joblib

## 2. Random Forest baseline (same features)

In [ ]:
!python -m models.m1_run --model rf --exp $EXP --data $DATA --save-model models/weights/rf.joblib

## 3. Ablations: CIC artefacts kept, query features only, shuffled split (S)

In [ ]:
!python -m models.m1_run --model m1 --exp $EXP --keep-cic-leaky --tag ablation_cic_leaky --data $DATA --no-shap
!python -m models.m1_run --model m1 --exp $EXP --features query --tag ablation_query_only --data $DATA --no-shap
!python -m models.m1_run --model m1 --exp S --split shuffled --data $DATA --no-shap

## 4. Quick look (official metrics come from the shared eval module)

In [ ]:
import json, glob, numpy as np
from sklearn.metrics import roc_auc_score
for f in sorted(glob.glob('results/[mr][1f]/*.json')):
    r = json.load(open(f)); y, s = np.array(r['labels']), np.array(r['scores'])
    auc = roc_auc_score(y, s) if 0 < y.sum() < len(y) else float('nan')
    print(f"{r['model']:22s} {r['experiment']:6s} AUC={auc:.4f} train={r['train_time_s']:.0f}s lat={r['latency_ms_per_query']:.2f}ms")

In [ ]:
from IPython.display import Image, display
for p in ['results/m1/shap/E2_beeswarm.png', 'results/m1/shap/E2_bar.png']:
    display(Image(p))

In [ ]:
# copy results + weights to Drive; push results/ to your branch from your laptop
!mkdir -p $DRIVE/results && cp -r results/m1 results/rf $DRIVE/results/ && cp -r models/weights $DRIVE/ && ls -R $DRIVE/results/m1 $DRIVE/results/rf | head